# E-commerce Conversion Analytics and Purchase Propensity ML

## tl;dr

- 310,014 sessions produced 4,247 purchases and a 1.37% conversion rate.
- The largest funnel loss occurs from Visit to Product View.
- The highest-scored 10% of future sessions captures 20.1% of purchases at 2.01x lift.
- This notebook reads the external `data/session_level_ecommerce.csv.gz` file and shows every analytical output and chart.

## Context & Methods

The notebook combines funnel analytics, a leakage-safe feature contract, chronological model validation, lift analysis, calibration, and experiment design. All narrative, code, tables, and charts are kept together for portfolio review.

### 1. Setup

In [1]:
from pathlib import Path
import hashlib
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2E74B5", "#B4872B", "#64748B", "#0F766E"]
print(f"pandas {pd.__version__} | NumPy {np.__version__}")

pandas 2.3.3 | NumPy 2.4.1


### 2. Load the external session-level CSV

In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "session_level_ecommerce.csv.gz").exists():
    PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "session_level_ecommerce.csv.gz"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Expected data/session_level_ecommerce.csv.gz. Run the notebook from the repository root "
        "or keep the original repository folder structure."
    )
df = pd.read_csv(DATA_PATH, parse_dates=["session_date"])
print(f"Loaded {len(df):,} sessions and {len(df.columns)} columns from {DATA_PATH}")
display(df.head())

Loaded 310,014 sessions and 30 columns from data/session_level_ecommerce.csv.gz


session_id,user_pseudo_id,ga_session_id,session_date,day_of_week,is_weekend,traffic_source,traffic_medium,device_category,operating_system,country,total_events,distinct_event_types,page_views,product_views,add_to_cart_events,checkout_events,shipping_events,payment_events,purchase_events,viewed_product,added_to_cart,began_checkout,added_shipping_info,added_payment_info,converted,transaction_count,revenue,session_duration_seconds,engagement_time_seconds
7802771.1242374269-5349183729,7.802771e+06,5349183729,2020-11-16,2,0,(direct),(none),mobile,Android,Turkey,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,928.531
48923044.1784847148-2085685405,4.892304e+07,2085685405,2020-11-16,2,0,(direct),(none),desktop,Web,United States,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,5.710
2342957.9635959151-4434005275,2.342958e+06,4434005275,2020-11-16,2,0,shop.googlemerchandisestore.com,referral,mobile,Web,India,2,2,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0.0,0.000
44249101.7248838631-8478097236,4.424910e+07,8478097236,2020-11-16,2,0,<Other>,<Other>,mobile,Web,(not set),2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.000
83459313.5629300570-8765897980,8.345931e+07,8765897980,2020-11-16,2,0,(data deleted),(data deleted),mobile,Web,United Kingdom,2,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.000


## Data

### 3. Validate input quality and the session key

In [3]:
required = [
    "session_id", "session_date", "day_of_week", "is_weekend", "traffic_source",
    "traffic_medium", "device_category", "operating_system", "country", "converted",
]
missing_columns = sorted(set(required) - set(df.columns))
quality = pd.DataFrame({
    "check": ["Required columns missing", "Missing values", "Duplicate session IDs", "Start date", "End date"],
    "result": [
        str(missing_columns), int(df[required].isna().sum().sum()),
        int(df["session_id"].duplicated().sum()), df["session_date"].min().date(),
        df["session_date"].max().date(),
    ],
})
assert not missing_columns
assert df["session_id"].is_unique
display(quality)
print("Data-quality checks passed.")

check,result
Required columns missing,[]
Missing values,0
Duplicate session IDs,0
Start date,2020-11-16
End date,2021-01-31


Data-quality checks passed.


### 4. Review business KPIs

In [4]:
kpis = pd.DataFrame({
    "KPI": ["Sessions", "Purchases", "Conversion rate", "Recorded revenue", "Revenue per session"],
    "Value": [
        f"{len(df):,}", f"{int(df['converted'].sum()):,}", f"{df['converted'].mean():.2%}",
        f"${df['revenue'].sum():,.0f}", f"${df['revenue'].sum()/len(df):.2f}",
    ],
})
display(kpis)


KPI,Value
Sessions,"310,014"
Purchases,"4,247"
Conversion rate,1.37%
Recorded revenue,"$315,948"
Revenue per session,$1.02


## Results

### 5. Build and visualize the conversion funnel

In [5]:
calibration = calibration_table(y_test, p_logistic)
display(calibration)
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(calibration["predicted_rate"]*100, calibration["observed_rate"]*100,
        marker="o", color=COLORS[0], linewidth=2, label="Model")
limit = max(calibration["predicted_rate"].max(), calibration["observed_rate"].max())*100*1.1
ax.plot([0, limit], [0, limit], linestyle="--", color="#64748B", label="Perfect calibration")
ax.set_title("Calibration by Equal-Frequency Risk Band", loc="left", fontsize=15, weight="bold", pad=14)
ax.set_xlabel("Predicted conversion rate (%)")
ax.set_ylabel("Observed conversion rate (%)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

stage,sessions,stage_conversion,drop_off
Visit,310014,1.0000,NaN
Product View,65393,0.2109,244621.0
Add to Cart,15187,0.2322,50206.0
Checkout,9086,0.5983,6101.0
Payment,5875,0.6466,3211.0
Purchase,4247,0.7229,1628.0


## Takeaways

The model offers modest but operationally useful ranking signal. It should be used to stratify randomized experiments, not to infer causality or exclude customers. Discovery and product-detail interventions deserve priority because the largest losses occur before cart and checkout.

### Reproducibility checks

In [12]:
assert len(df) == 310_014
assert int(df["converted"].sum()) == 4_247
assert len(train) == 228_487 and int(y_train.sum()) == 3_335
assert len(test) == 81_527 and int(y_test.sum()) == 912
assert 1.9 < results.loc["Hashed logistic regression", "Top-10% lift"] < 2.1
print("All reproducibility checks passed.")

All reproducibility checks passed.


### Limitations

- The observation window is short and includes seasonal behavior.
- Session-start context cannot represent product intent that appears later in a visit.
- Funnel findings are descriptive; intervention impact must be established experimentally.
- Model calibration and feature drift should be monitored before production use.